# 🛡️ AI-Powered Cyber Defense System: Threat Detection & Incident Mitigation
### Machine Learning for Automated Cybersecurity Telemetry Analysis
**Domain:** Applied Machine Learning & Cyber Threat Intelligence  
**Target Attacks:** Malware, Phishing, Ransomware, DDoS, and Zero-Day Infiltration  

---

## 1. Problem Statement & Architecture
Cyber attacks such as **malware**, **phishing**, and **ransomware** are increasing rapidly, making traditional rule-based and signature-driven security systems less effective. 

The **AI-Powered Cyber Defense System** utilizes Artificial Intelligence and Machine Learning to detect threats automatically and provide real-time protection against cyber attacks.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
%matplotlib inline
print('[+] Packages successfully loaded.')

## 2. Ingest Security Telemetry Dataset

In [ ]:
data_path = '../data/raw_cyber_security_telemetry.csv'
df = pd.read_csv(data_path)
print(f"Telemetry Shape: {df.shape}")
df.head()

In [ ]:
print("Missing values check:")
print(df.isnull().sum())
print("\nThreat Class Distribution:")
print(df['threat_label'].value_counts(normalize=True))

## 3. Exploratory Data Analysis & Security Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Shannon File Entropy (Ransomware Marker)
sns.boxplot(data=df, x='threat_label', y='file_entropy_score', ax=axes[0, 0], palette='Reds')
axes[0, 0].set_title('Shannon File Entropy (Encryption Metric)', fontweight='bold')

# 2. Crypto API Calls
sns.boxplot(data=df, x='threat_label', y='crypto_api_calls', ax=axes[0, 1], palette='Purples')
axes[0, 1].set_title('Cryptographic API Calls / sec', fontweight='bold')

# 3. SYN/ACK Ratio (DDoS Marker)
sns.boxplot(data=df, x='threat_label', y='syn_ack_ratio', ax=axes[1, 0], palette='Blues')
axes[1, 0].set_title('SYN/ACK Ratio (SYN Flood Detection)', fontweight='bold')

# 4. IP Reputation Score
sns.boxplot(data=df, x='threat_label', y='ip_reputation_score', ax=axes[1, 1], palette='Greens')
axes[1, 1].set_title('Threat Intelligence IP Reputation Score', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Preprocessing & Feature Transformation Pipeline

In [ ]:
import sys
sys.path.append('..')
from src.cyber_preprocessing import build_cyber_preprocessor, prepare_data_for_modeling

X, y, label_encoder = prepare_data_for_modeling(df, target_col='threat_label')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = build_cyber_preprocessor()
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print(f"Target Classes: {label_encoder.classes_}")
print(f"Transformed Training Matrix Shape: {X_train_trans.shape}")

## 5. Model Benchmarking & Multi-Algorithm Evaluation

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=16, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=120, max_depth=5, random_state=42),
    "SVM": SVC(kernel='rbf', C=1.5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

benchmark_summary = []
for name, model in models.items():
    model.fit(X_train_trans, y_train)
    y_pred = model.predict(X_test_trans)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    benchmark_summary.append({'Algorithm': name, 'Accuracy': acc, 'Weighted F1': f1})

bench_df = pd.DataFrame(benchmark_summary)
bench_df

## 6. Confusion Matrix for Random Forest

In [ ]:
rf_model = models["Random Forest"]
y_pred_rf = rf_model.predict(X_test_trans)
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix - Random Forest Cyber Classifier', fontweight='bold')
plt.xlabel('Predicted Threat Class')
plt.ylabel('Actual Threat Class')
plt.show()

## 7. Real-Time Autonomous Threat Mitigation & SOAR Playbook Execution

In [ ]:
from src.defense_engine import CyberDefenseEngine

engine = CyberDefenseEngine(models_dir="../models")
sample_telemetry = {
    'duration_sec': 8.2,
    'protocol': 'SMB',
    'dest_port': 445,
    'packet_count': 420,
    'byte_count': 180000,
    'src_bytes_ratio': 0.82,
    'failed_logins': 2,
    'cpu_usage_pct': 94.0,
    'ram_usage_pct': 88.0,
    'file_io_rate_mb': 210.0,
    'file_entropy_score': 7.78,
    'dns_query_rate': 15,
    'url_length': 45,
    'suspicious_keywords_count': 3,
    'connection_count_10m': 20,
    'syn_ack_ratio': 1.05,
    'privilege_escalation_flag': 1,
    'crypto_api_calls': 88,
    'ip_reputation_score': 0.18
}

result = engine.analyze_event(sample_telemetry)
print("=================== AI THREAT VERDICT ===================")
print(f"Threat Classification : {result['predicted_threat']} ({result['confidence_percentage']}%)")
print(f"Severity Level        : {result['severity_level']}")
print(f"Anomaly Risk Score    : {result['anomaly_score']}%")
print(f"Mitigation Playbook   : {result['mitigation_playbook']['title']}")
print(f"Firewall Rule Action  : {result['mitigation_playbook']['firewall_rule']}")
print("=========================================================")